In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
import matplotlib.colors as mcolors
from matplotlib.cm import ScalarMappable
from matplotlib.colors import ListedColormap
import geopandas as gpd

This notebook serves to compare historical values with forecasted values

In [2]:
# read circuit lengths
circuitlengths = pd.read_excel("//sce.eix.com/workgroup/SPE_DSA/Charles/IgnitionGoalSettingInput/Q1_2025-_HFRA_Conductor_SumStat_Circuit.xlsx")
# filter for OH only
circuitlengths = circuitlengths[circuitlengths['Conductor Type'].str.contains('OH',na=False)]
# filter for HFRA only
circuitlengths = circuitlengths[circuitlengths['HFRA Tier'].notna()]
# remove blank circuits
circuitlengths = circuitlengths[circuitlengths['Circuit Name'].notna()]
# capitalize circuit names
circuitlengths['Circuit Name'] = circuitlengths['Circuit Name'].str.upper()
circuitlengths_map = circuitlengths.groupby(['Circuit Name','Circuit FLOC'])['Circuit Miles'].sum().reset_index()
# get Circuit FLOC 
circuitlengths_map

,Circuit Name,Circuit FLOC,Circuit Miles
0,ABACUS,ED-00005,1.592161
1,ABRAHAM,ED-00024,1.204734
2,ACADEMY,ED-00045,14.998395
3,ACADIAN,ED-00032,3.743688
4,ACAPULCO,ED-00041,5.780607
...,...,...,...
1614,ZEVO,ED-19826,0.061934
1615,ZINC,ED-19890,7.881998
1616,ZIRCON,ED-19835,0.519011
1617,ZONE,ED-19850,44.549072


In [3]:
# read forecast distribution data
dist_df_forecast = pd.read_csv('//sce.eix.com/workgroup/SPE_DSA/Charles/IgnitionGoalSettingMidput/Dist_Forecasts_2025.csv', low_memory=False)
# make sure Date is date string of Datetime
dist_df_forecast['Date'] = pd.to_datetime(dist_df_forecast['Datetime']).dt.strftime('%Y-%m-%d')
# for forecast data, groupby Date, Circuit, take the mean of FuelsIndex_max_FPI2
dist_df_forecast_agg = dist_df_forecast.groupby(['Date', 'Circuit'])['FuelsIndex_max_FPI2'].mean().reset_index()

In [4]:
# for forecast data, groupby Date, Circuit, take the mean of FuelsIndex_max_FPI2
dist_df_forecast_agg = dist_df_forecast.groupby(['Date', 'Circuit'])['FuelsIndex_max_FPI2'].mean().reset_index()
# # for forecast data, groupby Date, Circuit, take the max of FuelsIndex_max_FPI2
# dist_df_forecast_agg = dist_df_forecast.groupby(['Date', 'Circuit'])['FuelsIndex_max_FPI2'].max().reset_index()
# merge with circuit lengths
dist_df_forecast_agg = dist_df_forecast_agg.merge(circuitlengths_map, left_on='Circuit', right_on='Circuit Name', how='left')
# locate rows where Circuit Miles is NaN and Circuit contains ambercrest and change value to 32.7754002840092
dist_df_forecast_agg.loc[dist_df_forecast_agg['Circuit'].str.contains('AMBERCREST', na=False) & dist_df_forecast_agg['Circuit Miles'].isna(), ['Circuit Name','Circuit Miles']] = ['AMBERCREST',32.7754002840092]
# locate rows where Circuit Miles is NaN and Circuit contains aventura and change value to 15.7032769637841
dist_df_forecast_agg.loc[dist_df_forecast_agg['Circuit'].str.contains('AVENTURA', na=False) & dist_df_forecast_agg['Circuit Miles'].isna(), ['Circuit Name','Circuit Miles']] = ['AVENTURA',15.7032769637841]
# locate rows where Circuit Miles is NaN and Circuit contains birchim and change value to 46.34484383
dist_df_forecast_agg.loc[dist_df_forecast_agg['Circuit'].str.contains('BIRCHIM', na=False) & dist_df_forecast_agg['Circuit Miles'].isna(), ['Circuit Name','Circuit Miles']] = ['BIRCHIM',46.34484383]
# locate rows where Circuit Miles is NaN and Circuit contains cafe and change value to 27.6402669615555
dist_df_forecast_agg.loc[dist_df_forecast_agg['Circuit'].str.contains('CAFE', na=False) & dist_df_forecast_agg['Circuit Miles'].isna(), ['Circuit Name','Circuit Miles']] = ['CAFE',27.6402669615555]
# locate rows where Circuit Miles is NaN and Circuit contains canebrake and change value to 66.16859376
dist_df_forecast_agg.loc[dist_df_forecast_agg['Circuit'].str.contains('CANEBRAKE', na=False) & dist_df_forecast_agg['Circuit Miles'].isna(), ['Circuit Name','Circuit Miles']] = ['CANEBRAKE',66.16859376]
# locate rows where Circuit Miles is NaN and Circuit contains casals and change value to 0.446999553409736
dist_df_forecast_agg.loc[dist_df_forecast_agg['Circuit'].str.contains('CASALS', na=False) & dist_df_forecast_agg['Circuit Miles'].isna(), ['Circuit Name','Circuit Miles']] = ['CASALS',0.446999553409736]
# locate rows where Circuit Miles is NaN and Circuit contains coachella and change value to 6.590021052
dist_df_forecast_agg.loc[dist_df_forecast_agg['Circuit'].str.contains('COACHELLA', na=False) & dist_df_forecast_agg['Circuit Miles'].isna(), ['Circuit Name','Circuit Miles']] = ['COACHELLA',6.590021052]
# locate rows where circuit miles is NaN and circuit contains energy and change value to 32.37798311
dist_df_forecast_agg.loc[dist_df_forecast_agg['Circuit'].str.contains('ENERGY', na=False) & dist_df_forecast_agg['Circuit Miles'].isna(), ['Circuit Name','Circuit Miles']] = ['ENERGY',32.37798311]
# locate rows where circuit miles is NaN and circuit contains fingal and change value to 36.6548594697613
dist_df_forecast_agg.loc[dist_df_forecast_agg['Circuit'].str.contains('FINGAL', na=False) & dist_df_forecast_agg['Circuit Miles'].isna(), ['Circuit Name','Circuit Miles']] = ['FINGAL',36.6548594697613]
# locate rows where circuit miles is NaN and circuit contains guitar and change value to 42.2491638220717
dist_df_forecast_agg.loc[dist_df_forecast_agg['Circuit'].str.contains('GUITAR', na=False) & dist_df_forecast_agg['Circuit Miles'].isna(), ['Circuit Name','Circuit Miles']] = ['GUITAR',42.2491638220717]
# locate rows where circuit miles is NaN and circuit contains huckleberry and change value to 25.4840533042463
dist_df_forecast_agg.loc[dist_df_forecast_agg['Circuit'].str.contains('HUCKLEBERRY', na=False) & dist_df_forecast_agg['Circuit Miles'].isna(), ['Circuit Name','Circuit Miles']] = ['HUCKLEBERRY',25.4840533042463]
# locate rows where circuit miles is NaN and circuit contains Kinsey and change value to 38.2093922
dist_df_forecast_agg.loc[dist_df_forecast_agg['Circuit'].str.contains('KINSEY', na=False) & dist_df_forecast_agg['Circuit Miles'].isna(), ['Circuit Name','Circuit Miles']] = ['KINSEY',38.2093922]
# locate rows where circuit miles is NaN and circuit contains Penstock and change value to 50.90351311
dist_df_forecast_agg.loc[dist_df_forecast_agg['Circuit'].str.contains('PENSTOCK', na=False) & dist_df_forecast_agg['Circuit Miles'].isna(), ['Circuit Name','Circuit Miles']] = ['PENSTOCK',50.90351311]
# locate rows where circuit miles is NaN and circuit contains Shovel and change value to 47.51064986
dist_df_forecast_agg.loc[dist_df_forecast_agg['Circuit'].str.contains('SHOVEL', na=False) & dist_df_forecast_agg['Circuit Miles'].isna(), ['Circuit Name','Circuit Miles']] = ['SHOVEL',47.51064986]
# locate rows where circuit miles is NaN and circuit contains Tejon and change value to 57.8330462
dist_df_forecast_agg.loc[dist_df_forecast_agg['Circuit'].str.contains('TEJON', na=False) & dist_df_forecast_agg['Circuit Miles'].isna(), ['Circuit Name','Circuit Miles']] = ['TEJON',57.8330462]
# locate rows where circuit miles is NaN and circuit contains Toll and change value to 8.228283341
dist_df_forecast_agg.loc[dist_df_forecast_agg['Circuit'].str.contains('TOLL', na=False) & dist_df_forecast_agg['Circuit Miles'].isna(), ['Circuit Name','Circuit Miles']] = ['TOLL',8.228283341]
# locate rows where circuit miles is NaN and circuit contains Sutt and change value to 16.8635623
dist_df_forecast_agg.loc[dist_df_forecast_agg['Circuit'].str.contains('SUTT', na=False) & dist_df_forecast_agg['Circuit Miles'].isna(), ['Circuit Name','Circuit Miles']] = ['SUTT',16.8635623]
# locate rows where circuit miles is NaN and circuit contains POPPET FLATS and change value to 31.30335437
dist_df_forecast_agg.loc[dist_df_forecast_agg['Circuit'].str.contains('POPPET FLATS', na=False) & dist_df_forecast_agg['Circuit Miles'].isna(), ['Circuit Name','Circuit Miles']] = ['POPPET FLATS',31.30335437]
# sort values by Date and Circuit Name and FuelsIndex_max_FPI2 descending
dist_df_forecast_agg = dist_df_forecast_agg.sort_values(by=['Date', 'Circuit Name', 'FuelsIndex_max_FPI2'], ascending=[True, True, False])
# # drop duplicate Date and Circuit Name, keep first one
dist_df_forecast_agg = dist_df_forecast_agg.drop_duplicates(subset=['Date', 'Circuit Name'], keep='first')
dist_df_forecast_agg

,Date,Circuit,FuelsIndex_max_FPI2,Circuit Name,Circuit FLOC,Circuit Miles
0,2025-01-01,ABACUS,0.47875,ABACUS,ED-00005,1.592161
1,2025-01-01,ABRAHAM,0.66875,ABRAHAM,ED-00024,1.204734
2,2025-01-01,ACADEMY,0.65000,ACADEMY,ED-00045,14.998395
3,2025-01-01,ACADIAN,0.62500,ACADIAN,ED-00032,3.743688
4,2025-01-01,ACAPULCO,0.51500,ACAPULCO,ED-00041,5.780607
...,...,...,...,...,...,...
358551,2025-11-10,ZEVO,0.57625,ZEVO,ED-19826,0.061934
358552,2025-11-10,ZINC,0.59250,ZINC,ED-19890,7.881998
358553,2025-11-10,ZIRCON,0.60000,ZIRCON,ED-19835,0.519011
358554,2025-11-10,ZONE,0.66000,ZONE,ED-19850,44.549072


In [5]:
dist_df_forecast_agg

,Date,Circuit,FuelsIndex_max_FPI2,Circuit Name,Circuit FLOC,Circuit Miles
0,2025-01-01,ABACUS,0.47875,ABACUS,ED-00005,1.592161
1,2025-01-01,ABRAHAM,0.66875,ABRAHAM,ED-00024,1.204734
2,2025-01-01,ACADEMY,0.65000,ACADEMY,ED-00045,14.998395
3,2025-01-01,ACADIAN,0.62500,ACADIAN,ED-00032,3.743688
4,2025-01-01,ACAPULCO,0.51500,ACAPULCO,ED-00041,5.780607
...,...,...,...,...,...,...
358551,2025-11-10,ZEVO,0.57625,ZEVO,ED-19826,0.061934
358552,2025-11-10,ZINC,0.59250,ZINC,ED-19890,7.881998
358553,2025-11-10,ZIRCON,0.60000,ZIRCON,ED-19835,0.519011
358554,2025-11-10,ZONE,0.66000,ZONE,ED-19850,44.549072


In [6]:
# read historical distribution data
dist_df_historical = pd.read_csv(r"\\sce.eix.com\workgroup\SPE_DSA\Charles\IgnitionGoalSettingMidput\Historical_Fuel_01082025.csv", low_memory=False)
dist_df_historical['Date'] = pd.to_datetime(dist_df_historical['time']).dt.strftime('%Y-%m-%d')
# rename id to Circuit
dist_df_historical = dist_df_historical.rename(columns={'id': 'Circuit'})
fuel_var = 'mean_max_fpi_2.0_tdd_fuels_index_om'

In [7]:
# merge the two dataframes on Date and Circuit
merged_df = pd.merge(dist_df_forecast_agg, dist_df_historical[['Date', 'Circuit', fuel_var]], on=['Date', 'Circuit'], how='inner', suffixes=('_forecast', '_historical'))
# calculate the difference
merged_df['FuelsIndex_Difference'] = merged_df['FuelsIndex_max_FPI2'] - merged_df[fuel_var]
# calculate the absolute difference
merged_df['FuelsIndex_AbsDifference'] = merged_df['FuelsIndex_Difference'].abs()
merged_df

,Date,Circuit,FuelsIndex_max_FPI2,Circuit Name,Circuit FLOC,Circuit Miles,mean_max_fpi_2.0_tdd_fuels_index_om,FuelsIndex_Difference,FuelsIndex_AbsDifference
0,2025-01-01,ABACUS,0.478750,ABACUS,ED-00005,1.592161,0.609913,-0.131163,0.131163
1,2025-01-01,ABRAHAM,0.668750,ABRAHAM,ED-00024,1.204734,0.748564,-0.079814,0.079814
2,2025-01-01,ACADEMY,0.650000,ACADEMY,ED-00045,14.998395,0.639831,0.010169,0.010169
3,2025-01-01,ACADIAN,0.625000,ACADIAN,ED-00032,3.743688,0.723367,-0.098367,0.098367
4,2025-01-01,ACAPULCO,0.515000,ACAPULCO,ED-00041,5.780607,0.551451,-0.036451,0.036451
...,...,...,...,...,...,...,...,...,...
8251,2025-01-08,ZEVO,0.660357,ZEVO,ED-19826,0.061934,0.730694,-0.070336,0.070336
8252,2025-01-08,ZINC,0.754107,ZINC,ED-19890,7.881998,0.766034,-0.011926,0.011926
8253,2025-01-08,ZIRCON,0.763393,ZIRCON,ED-19835,0.519011,0.758810,0.004583,0.004583
8254,2025-01-08,ZONE,0.780357,ZONE,ED-19850,44.549072,0.797821,-0.017464,0.017464


In [8]:
# get descriptive statistics of the difference
merged_df['FuelsIndex_Difference'].describe()

count    8248.000000
mean       -0.049154
std         0.057749
min        -0.561818
25%        -0.078232
50%        -0.047018
75%        -0.013717
max         0.246520
Name: FuelsIndex_Difference, dtype: float64

In [9]:
# get descriptive statistics of the absolute difference
merged_df['FuelsIndex_AbsDifference'].describe()

count    8248.000000
mean        0.058194
std         0.048625
min         0.000003
25%         0.023360
50%         0.049339
75%         0.079438
max         0.561818
Name: FuelsIndex_AbsDifference, dtype: float64

In [10]:
# find the top 10 largest absolute differences
merged_df.nlargest(10, 'FuelsIndex_AbsDifference')

,Date,Circuit,FuelsIndex_max_FPI2,Circuit Name,Circuit FLOC,Circuit Miles,mean_max_fpi_2.0_tdd_fuels_index_om,FuelsIndex_Difference,FuelsIndex_AbsDifference
6203,2025-01-07,AGNEW,0.201509,AGNEW,ED-00142,0.372587,0.763328,-0.561818,0.561818
5171,2025-01-06,AGNEW,0.203542,AGNEW,ED-00142,0.372587,0.754254,-0.550712,0.550712
7235,2025-01-08,AGNEW,0.214821,AGNEW,ED-00142,0.372587,0.765225,-0.550404,0.550404
4139,2025-01-05,AGNEW,0.212000,AGNEW,ED-00142,0.372587,0.729205,-0.517205,0.517205
2075,2025-01-03,AGNEW,0.227083,AGNEW,ED-00142,0.372587,0.724886,-0.497803,0.497803
3107,2025-01-04,AGNEW,0.214688,AGNEW,ED-00142,0.372587,0.701015,-0.486328,0.486328
5301,2025-01-06,CAIN RANCH,0.103125,CAIN RANCH,ED-02637,0.228991,0.583105,-0.479980,0.479980
7365,2025-01-08,CAIN RANCH,0.109286,CAIN RANCH,ED-02637,0.228991,0.580110,-0.470824,0.470824
6333,2025-01-07,CAIN RANCH,0.103208,CAIN RANCH,ED-02637,0.228991,0.573778,-0.470571,0.470571
1043,2025-01-02,AGNEW,0.225625,AGNEW,ED-00142,0.372587,0.694740,-0.469115,0.469115


# Looking at FPI 1.0 Forecast 2024 data

In [11]:
# read distribution data
distribution_data = pd.read_csv('//sce.eix.com/workgroup/SPE_DSA/Charles/IgnitionGoalSettingOutput/CircuitData_Dashboard.csv',low_memory=False)
# keep only Year 2024
distribution_data = distribution_data[distribution_data['Year'] == 2024]
distribution_data

,Date,Year,Circuit Name,mean_max_fpi_2.0_tdd_fuels_index,Circuit FLOC,Circuit Miles,Fuel Category
3287,2024-01-01,2024,HI LINE,0.159343,ED-08427,25.181100,Low
3288,2024-01-02,2024,HI LINE,0.175226,ED-08427,25.181100,Low
3289,2024-01-03,2024,HI LINE,0.193372,ED-08427,25.181100,Low
3290,2024-01-04,2024,HI LINE,0.198691,ED-08427,25.181100,Low
3291,2024-01-05,2024,HI LINE,0.234902,ED-08427,25.181100,Low
...,...,...,...,...,...,...,...
4043866,2024-12-27,2024,STROSNIDER,0.306157,ED-17122,15.212386,Low
4043867,2024-12-28,2024,STROSNIDER,0.286921,ED-17122,15.212386,Low
4043868,2024-12-29,2024,STROSNIDER,0.277896,ED-17122,15.212386,Low
4043869,2024-12-30,2024,STROSNIDER,0.304461,ED-17122,15.212386,Low


In [12]:
distribution_data[distribution_data['Fuel Category']=='High']

,Date,Year,Circuit Name,mean_max_fpi_2.0_tdd_fuels_index,Circuit FLOC,Circuit Miles,Fuel Category
3473,2024-07-05,2024,HI LINE,0.854502,ED-08427,25.181100,High
3474,2024-07-06,2024,HI LINE,0.875685,ED-08427,25.181100,High
3478,2024-07-10,2024,HI LINE,0.871506,ED-08427,25.181100,High
3479,2024-07-11,2024,HI LINE,0.878409,ED-08427,25.181100,High
3480,2024-07-12,2024,HI LINE,0.890293,ED-08427,25.181100,High
...,...,...,...,...,...,...,...
4043695,2024-07-09,2024,STROSNIDER,0.859241,ED-17122,15.212386,High
4043696,2024-07-10,2024,STROSNIDER,0.867661,ED-17122,15.212386,High
4043697,2024-07-11,2024,STROSNIDER,0.878699,ED-17122,15.212386,High
4043698,2024-07-12,2024,STROSNIDER,0.879649,ED-17122,15.212386,High


In [13]:
# read old fpi forecast data
old_forecast_folder = r"\\sce.eix.com\workgroup\SPE_DSA\Charles\2024_Forecast_Data\2024 files"
date_range = pd.date_range(start='2024-01-01', end='2024-12-31', freq='D')
# loop through each date and read the corresponding csv file
old_forecast_list = []
for date in date_range:
    date_str = date.strftime('%Y-%m-%d')
    file_path = rf"{old_forecast_folder}\Distribution_{date_str}_06.csv"
    try:
        df = pd.read_csv(file_path, low_memory=False)
        df['Date'] = pd.to_datetime(df['Datetime']).dt.strftime('%Y-%m-%d')
        # keep rows with Date equal to date_str
        df = df[df['Date'] == date_str]
        old_forecast_list.append(df[['Date', 'Circuit', 'DL', 'G', 'LFM', 'F10']])
    except FileNotFoundError:
        print(f"File not found: {file_path}")

In [14]:
distribution_data

,Date,Year,Circuit Name,mean_max_fpi_2.0_tdd_fuels_index,Circuit FLOC,Circuit Miles,Fuel Category
3287,2024-01-01,2024,HI LINE,0.159343,ED-08427,25.181100,Low
3288,2024-01-02,2024,HI LINE,0.175226,ED-08427,25.181100,Low
3289,2024-01-03,2024,HI LINE,0.193372,ED-08427,25.181100,Low
3290,2024-01-04,2024,HI LINE,0.198691,ED-08427,25.181100,Low
3291,2024-01-05,2024,HI LINE,0.234902,ED-08427,25.181100,Low
...,...,...,...,...,...,...,...
4043866,2024-12-27,2024,STROSNIDER,0.306157,ED-17122,15.212386,Low
4043867,2024-12-28,2024,STROSNIDER,0.286921,ED-17122,15.212386,Low
4043868,2024-12-29,2024,STROSNIDER,0.277896,ED-17122,15.212386,Low
4043869,2024-12-30,2024,STROSNIDER,0.304461,ED-17122,15.212386,Low


In [15]:
# loop through each file in list, aggregate by Circuit, take mean of DL, G, LFM, F10
old_forecast_agg_list = []
for df in old_forecast_list:
    agg_df = df.groupby('Circuit')[['DL', 'G', 'LFM', 'F10']].mean().reset_index()
    agg_df['Date'] = df['Date'].iloc[0]
    old_forecast_agg_list.append(agg_df)
# concatenate all dataframes in the list
old_forecast_agg = pd.concat(old_forecast_agg_list, ignore_index=True)
# calculate fuel part by taking DL/LFM + G
old_forecast_agg['FuelsIndex_max_FPI2'] = (old_forecast_agg['DL'] / old_forecast_agg['LFM']) + old_forecast_agg['G']
# get the max value in df
max_fuels_index_agg = old_forecast_agg['FuelsIndex_max_FPI2'].max()
# get the percent value
old_forecast_agg['FuelsIndex_max_FPI2_Percent'] = old_forecast_agg['FuelsIndex_max_FPI2'] / 10.5
# get the circuit lengths by merging with distribution_data
old_forecast_agg = pd.merge(old_forecast_agg, distribution_data[['Circuit Name', 'mean_max_fpi_2.0_tdd_fuels_index','Circuit Miles']].drop_duplicates('Circuit Name'), left_on='Circuit', right_on='Circuit Name', how='inner')
# categorize Fuel Category as High if FuelsIndex_max_FPI2_Percent >= 0.85, Low otherwise
old_forecast_agg['Fuel Category'] = np.where(old_forecast_agg['FuelsIndex_max_FPI2_Percent'] >= 0.85, 'High', 'Low')
# calculate absolute difference between FuelsIndex_max_FPI2_Percent and mean_max_fpi_2.0_tdd_fuels_index
old_forecast_agg['FuelsIndex_AbsDifference'] = (old_forecast_agg['FuelsIndex_max_FPI2_Percent'] - old_forecast_agg['mean_max_fpi_2.0_tdd_fuels_index']).abs()
# calculate difference between FuelsIndex_max_FPI2_Percent and mean_max_fpi_2.0_tdd_fuels_index
old_forecast_agg['FuelsIndex_Difference'] = old_forecast_agg['FuelsIndex_max_FPI2_Percent'] - old_forecast_agg['mean_max_fpi_2.0_tdd_fuels_index']
# calculate ratio between FuelsIndex_max_FPI2 and mean_max_fpi_2.0_tdd_fuels_index
old_forecast_agg['FuelsIndex_Ratio'] = old_forecast_agg['mean_max_fpi_2.0_tdd_fuels_index'] / old_forecast_agg['FuelsIndex_max_FPI2_Percent']
# get adjusted FuelsIndex_max_FPI2 by multiplying by 0.5
old_forecast_agg['FuelsIndex_max_FPI2_Adjusted'] = old_forecast_agg['FuelsIndex_max_FPI2_Percent'] * 0.5
# calculate the difference between adjusted and mean_max_fpi_2.0_tdd_fuels_index
old_forecast_agg['FuelsIndex_Adjusted_Difference'] = old_forecast_agg['FuelsIndex_max_FPI2_Adjusted'] - old_forecast_agg['mean_max_fpi_2.0_tdd_fuels_index']
# calculate absolute difference between adjusted and mean_max_fpi_2.0_tdd_fuels_index
old_forecast_agg['FuelsIndex_Adjusted_AbsDifference'] = old_forecast_agg['FuelsIndex_Adjusted_Difference'].abs()
# get fuel category based on adjusted fuels index
old_forecast_agg['Fuel Category Adjusted'] = np.where(old_forecast_agg['FuelsIndex_max_FPI2_Adjusted'] >= 0.85, 'High', 'Low')
old_forecast_agg

,Circuit,DL,G,LFM,F10,Date,FuelsIndex_max_FPI2,FuelsIndex_max_FPI2_Percent,Circuit Name,mean_max_fpi_2.0_tdd_fuels_index,Circuit Miles,Fuel Category,FuelsIndex_AbsDifference,FuelsIndex_Difference,FuelsIndex_Ratio,FuelsIndex_max_FPI2_Adjusted,FuelsIndex_Adjusted_Difference,FuelsIndex_Adjusted_AbsDifference,Fuel Category Adjusted
0,ABACUS,1.0,4.0,0.77750,0.19250,2024-01-01,5.286174,0.503445,ABACUS,0.341563,1.592161,Low,0.161883,0.161883,0.678450,0.251723,-0.089840,0.089840,Low
1,ABRAHAM,1.0,4.0,0.74750,0.23875,2024-01-01,5.337793,0.508361,ABRAHAM,0.291904,1.204734,Low,0.216458,0.216458,0.574205,0.254181,-0.037723,0.037723,Low
2,ACADEMY,1.0,4.0,0.77750,0.20500,2024-01-01,5.286174,0.503445,ACADEMY,0.290061,14.998395,Low,0.213384,0.213384,0.576152,0.251723,-0.038338,0.038338,Low
3,ACADIAN,1.0,4.0,0.72875,0.22750,2024-01-01,5.372213,0.511639,ACADIAN,0.267792,3.743688,Low,0.243847,0.243847,0.523401,0.255820,-0.011973,0.011973,Low
4,ACAPULCO,1.0,5.0,0.73750,0.27000,2024-01-01,6.355932,0.605327,ACAPULCO,0.249692,5.780607,Low,0.355635,0.355635,0.412492,0.302663,0.052971,0.052971,Low
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
376211,ZENDA,1.0,5.0,0.70875,0.18375,2024-12-31,6.410935,0.610565,ZENDA,0.214564,48.447842,Low,0.396001,0.396001,0.351419,0.305283,0.090718,0.090718,Low
376212,ZEVO,1.0,5.0,0.65000,0.17000,2024-12-31,6.538462,0.622711,ZEVO,0.237793,0.061934,Low,0.384917,0.384917,0.381868,0.311355,0.073562,0.073562,Low
376213,ZINC,1.0,5.0,0.65000,0.15500,2024-12-31,6.538462,0.622711,ZINC,0.207984,7.881998,Low,0.414726,0.414726,0.333998,0.311355,0.103371,0.103371,Low
376214,ZIRCON,1.0,5.0,0.65000,0.15500,2024-12-31,6.538462,0.622711,ZIRCON,0.207984,0.519011,Low,0.414726,0.414726,0.333998,0.311355,0.103371,0.103371,Low


In [16]:
mask = old_forecast_agg['Fuel Category Adjusted']=='High'
old_forecast_agg[mask]['Circuit Miles'].sum()

np.float64(0.0)

In [17]:
old_forecast_agg['FuelsIndex_Adjusted_Difference'].describe()

count    375850.000000
mean         -0.014369
std           0.138360
min          -0.430311
25%          -0.118359
50%           0.020043
75%           0.087388
max           0.408947
Name: FuelsIndex_Adjusted_Difference, dtype: float64

In [18]:
old_forecast_agg['FuelsIndex_Adjusted_AbsDifference'].describe()

count    375850.000000
mean          0.112773
std           0.081439
min           0.000001
25%           0.046673
50%           0.095551
75%           0.165681
max           0.430311
Name: FuelsIndex_Adjusted_AbsDifference, dtype: float64

In [19]:
old_forecast_agg['FuelsIndex_Ratio'].describe()

count    375850.000000
mean          1.064452
std           1.264977
min           0.060132
25%           0.372800
50%           0.468411
75%           0.987850
max           7.410037
Name: FuelsIndex_Ratio, dtype: float64

In [20]:
max_fuels_index_agg

np.float64(10.357142857142858)

In [21]:
old_forecast_agg[old_forecast_agg['Fuel Category']=='High']['Circuit Miles'].sum()

np.float64(169305.1317227982)

In [22]:
old_forecast_agg['FuelsIndex_max_FPI2_Percent'].describe()

count    376216.000000
mean          0.498594
std           0.256741
min           0.052365
25%           0.262626
50%           0.614217
75%           0.697674
max           0.985258
Name: FuelsIndex_max_FPI2_Percent, dtype: float64

In [23]:
old_forecast_agg['FuelsIndex_AbsDifference'].describe()

count    375850.000000
mean          0.311262
std           0.156139
min           0.000003
25%           0.187416
50%           0.335077
75%           0.428106
max           0.873800
Name: FuelsIndex_AbsDifference, dtype: float64

# Look at 2025 first half historical data

In [24]:
historical_2025_first_half = pd.read_csv(r"\\sce.eix.com\workgroup\SPE_DSA\Charles\IgnitionGoalSettingMidput\Historical_Fuel_07012025.csv", low_memory=False)
historical_2025_first_half['Date'] = pd.to_datetime(historical_2025_first_half['time']).dt.strftime('%Y-%m-%d')
historical_2025_first_half = historical_2025_first_half[['Date','id','mean_max_fpi_2.0_tdd_fuels_index',]]
# circuitlengths_dict = dist_df_forecast_agg.set_index('Circuit')['Circuit Miles'].to_dict()
# historical_2025_first_half['Circuit Miles'] = historical_2025_first_half['id'].map(circuitlengths_dict)
# merge with dist_df_forecast_agg on Date and Circuit
historical_2025_first_half = pd.merge(dist_df_forecast_agg, historical_2025_first_half, left_on=['Date', 'Circuit'], right_on=['Date', 'id'], how='inner')
historical_2025_first_half

,Date,Circuit,FuelsIndex_max_FPI2,Circuit Name,Circuit FLOC,Circuit Miles,id,mean_max_fpi_2.0_tdd_fuels_index
0,2025-01-01,ABACUS,0.478750,ABACUS,ED-00005,1.592161,ABACUS,0.609913
1,2025-01-01,ABRAHAM,0.668750,ABRAHAM,ED-00024,1.204734,ABRAHAM,0.748564
2,2025-01-01,ACADEMY,0.650000,ACADEMY,ED-00045,14.998395,ACADEMY,0.639831
3,2025-01-01,ACADIAN,0.625000,ACADIAN,ED-00032,3.743688,ACADIAN,0.723367
4,2025-01-01,ACAPULCO,0.515000,ACAPULCO,ED-00041,5.780607,ACAPULCO,0.551451
...,...,...,...,...,...,...,...,...
156998,2025-06-30,YUCATAN,0.842456,YUCATAN,ED-19788,2.112280,YUCATAN,0.853282
156999,2025-06-30,ZEVO,0.840526,ZEVO,ED-19826,0.061934,ZEVO,0.869785
157000,2025-06-30,ZINC,0.792105,ZINC,ED-19890,7.881998,ZINC,0.828223
157001,2025-06-30,ZIRCON,0.795088,ZIRCON,ED-19835,0.519011,ZIRCON,0.828223


In [25]:
mask1 = historical_2025_first_half['mean_max_fpi_2.0_tdd_fuels_index'] >= 0.85
print(historical_2025_first_half[mask1]['Circuit Miles'].sum())
mask2 = historical_2025_first_half['FuelsIndex_max_FPI2'] >= 0.85
print(historical_2025_first_half[mask2]['Circuit Miles'].sum())

152501.5341900672
51605.84293641668


In [26]:
# calculate the difference
historical_2025_first_half['FuelsIndex_Difference'] = historical_2025_first_half['FuelsIndex_max_FPI2'] - historical_2025_first_half['mean_max_fpi_2.0_tdd_fuels_index']
historical_2025_first_half['FuelsIndex_Difference'].describe()

count    156822.000000
mean         -0.052679
std           0.079793
min          -0.638331
25%          -0.087268
50%          -0.045156
75%          -0.006170
max           0.303471
Name: FuelsIndex_Difference, dtype: float64

In [27]:
# calculate the absolute difference
historical_2025_first_half['FuelsIndex_AbsDifference'] = historical_2025_first_half['FuelsIndex_Difference'].abs()
historical_2025_first_half['FuelsIndex_AbsDifference'].describe()

count    1.568220e+05
mean     6.949953e-02
std      6.566377e-02
min      4.454977e-07
25%      2.559726e-02
50%      5.341558e-02
75%      9.189611e-02
max      6.383314e-01
Name: FuelsIndex_AbsDifference, dtype: float64

In [28]:
historical_2025_first_half[historical_2025_first_half['mean_max_fpi_2.0_tdd_fuels_index'] >= 0.85]['Circuit Miles'].sum()

np.float64(152501.5341900672)